# Applying integrative multi-omics to the positive control case: D-gene

In [12]:
import pandas as pd
import numpy as np
import itertools

## VCF file for forward genetics
Opening and analysing the variant file containing variant sites exclusive to SBC4, 10, 23, samples that show "juiciness" phenotype.

In [3]:
%%bash
VCF_PATH=../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz
../docker/run.sh python3 ../workflow/scripts/genomics_scoring.py -i $VCF_PATH -o ../results/gene_score/ -g ../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz

[genomics_scoring] Parsing VCF: ../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz
[genomics_scoring]   205,996 (variant, gene) rows
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.variant.tsv  (205,996 variant annotations)
[genomics_scoring] Loading gene lengths: ../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz
[genomics_scoring]   32,458 genes with length
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.scores_summary.png
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.variant_scores.png


### Choosing the most appropriate merging method out of the four
Cost function: which method places D-gene on the highest rank?

In [4]:
df_scored               = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.variant.tsv", sep="\t")
df_scored_gene_max      = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv", sep="\t")
df_scored_gene_sum      = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv", sep="\t") 
df_scored_gene_max_norm = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv", sep="\t")
df_scored_gene_sum_norm = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv", sep="\t")

In [24]:
# Confirmed dPCD homologs in sorghum (Olvera-Carrillo 2015; ATTED-II orthologs).
# Keys are ann_gene_id (= "LOC<EGI>") as used in the gene-score tables.
DPCD_GENES = {
    "LOC8058198": "PASPA3-a", "LOC110430277": "PASPA3-b", "LOC110433911": "PASPA3-c",
    "LOC8058001": "CEP1/SEN102", "LOC8074232": "SCPL48",
    "LOC8063515": "RNS3-a", "LOC8063317": "RNS3-b", "LOC8077825": "RNS3-c",
    "LOC8084661": "D-gene (TF / positive control)",
}

def where_genes(df, genes=DPCD_GENES):
    """Rank & percentile of a gene set within a gene-score table.

    df    : gene-level score table keyed by `ann_gene_id` (= "LOC<EGI>"),
            e.g. df_scored_gene_max; must carry `rank`, `percentile`.
    genes : {ann_gene_id: label}. A gene absent from df gets a NaN row
            (= it carries no variant in this sharing group).
    Returns one row per gene in `genes`, sorted by rank (found genes first).
    """
    cols = [c for c in ["score", "percentile", "rank"] if c in df.columns]
    out = (df[df["ann_gene_id"].isin(genes)]
           .drop_duplicates("ann_gene_id")
           .set_index("ann_gene_id")[cols]
           .reindex(genes.keys()))
    out.insert(0, "gene", out.index.map(genes))
    out.insert(1, "found", out["rank"].notna())
    return out.sort_values("rank", na_position="last")

In [25]:
merged_dfs = {
    "gene_max":      df_scored_gene_max,
    "gene_sum":      df_scored_gene_sum,
    "gene_max_norm": df_scored_gene_max_norm,
    "gene_sum_norm": df_scored_gene_sum_norm,
}

# Percentile of every dPCD gene under each merging method (NaN = absent from group)
dpcd_percentile = pd.DataFrame(
    {name: where_genes(df)["percentile"] for name, df in merged_dfs.items()}
)
dpcd_percentile.insert(0, "gene", dpcd_percentile.index.map(DPCD_GENES))

n_genes = len(df_scored_gene_max)
found_any = dpcd_percentile.drop(columns="gene").notna().any(axis=1).sum()
print(f"dPCD genes found in the SBC4_SBC10_SBC23 group ({n_genes:,} genes): "
      f"{found_any}/{len(DPCD_GENES)}")
print("Percentile per merging method (higher = more strongly variant-affected):")
dpcd_percentile

dPCD genes found in the SBC4_SBC10_SBC23 group (12,917 genes): 2/9
Percentile per merging method (higher = more strongly variant-affected):


,gene,gene_max,gene_sum,gene_max_norm,gene_sum_norm
ann_gene_id,,,,,
LOC8058001,CEP1/SEN102,91.553766,91.000232,90.70827,94.408394
LOC8063515,RNS3-a,45.064643,15.456375,74.50642,40.480464
LOC8058198,PASPA3-a,NaN,NaN,NaN,NaN
LOC110430277,PASPA3-b,NaN,NaN,NaN,NaN
LOC110433911,PASPA3-c,NaN,NaN,NaN,NaN
LOC8074232,SCPL48,NaN,NaN,NaN,NaN
LOC8063317,RNS3-b,NaN,NaN,NaN,NaN
LOC8077825,RNS3-c,NaN,NaN,NaN,NaN
LOC8084661,D-gene (TF / positive control),NaN,NaN,NaN,NaN


## DMR analysis for forward genetics

In [16]:
from pathlib import Path

# DMR_annotated.tsv schema (one row per DMR x overlapping feature x gene):
#   chr, start, end, diff.Methy, direction (= "hyper_<accession>"),
#   sample_a, sample_b, feature (promoter|exon|CDS|intron|intergenic), gene_label
DMR_PATH = Path("../results/DMR/DMR_annotated.tsv")

df_dmr = pd.read_csv(DMR_PATH, sep="\t")
print(f"{len(df_dmr):,} DMR-feature rows | features: {sorted(df_dmr['feature'].unique())}")

# --- Candidate genes: promoter HYPOMETHYLATED in the dry accession SBC11 -------
# SBC11 is the lone dry accession. A promoter LESS methylated in SBC11 than in a
# juicy accession is de-repressed in SBC11 / silenced in the juicy ones -- the
# expression contrast that could underlie the juicy-vs-dry split. So candidates
# are genes whose promoter is hypomethylated in SBC11.
#
# Direction encoding: `direction` names the HYPER accession ("hyper_<acc>").
# Hypomethylated in SBC11  <=>  the contrast involves SBC11 AND the hyper side is
# the OTHER accession, i.e. direction != "hyper_SBC11".
SBC11 = "SBC11"

promoter_dmr = df_dmr[
    (df_dmr["feature"] == "promoter")
    & df_dmr["gene_label"].notna()
    & (df_dmr["gene_label"].astype(str).str.strip() != "")
].copy()

involves_sbc11 = (promoter_dmr["sample_a"] == SBC11) | (promoter_dmr["sample_b"] == SBC11)
hypo_in_sbc11  = involves_sbc11 & (promoter_dmr["direction"] != f"hyper_{SBC11}")
sbc11_hypo = promoter_dmr[hypo_in_sbc11].copy()
sbc11_hypo["abs_diff"] = sbc11_hypo["diff.Methy"].abs()

# one row per gene: its strongest SBC11-hypomethylated promoter DMR
promoter_candidates = (
    sbc11_hypo
    .sort_values("abs_diff", ascending=False)
    .drop_duplicates(subset="gene_label")
    .loc[:, ["gene_label", "diff.Methy", "abs_diff", "direction",
             "sample_a", "sample_b", "chr", "start", "end"]]
    .reset_index(drop=True)
)
print(f"{int(hypo_in_sbc11.sum()):,} promoter DMRs hypomethylated in {SBC11} -> "
      f"{len(promoter_candidates):,} candidate genes")
promoter_candidates.head(20)

152,253 DMR-feature rows | features: ['CDS', 'exon', 'intergenic', 'promoter']
6,961 promoter DMRs hypomethylated in SBC11 -> 5,367 candidate genes


,gene_label,diff.Methy,abs_diff,direction,sample_a,sample_b,chr,start,end
0,LOC110432410,-0.621911,0.621911,hyper_SBC23,SBC11,SBC23,NC_012871.2,9293745,9293880
1,LOC110432674,-0.618042,0.618042,hyper_SBC4,SBC11,SBC4,NC_012871.2,36039726,36039979
2,LOC110436852,-0.569947,0.569947,hyper_SBC4,SBC11,SBC4,NC_012876.2,63484731,63485968
3,LOC110437322,-0.569947,0.569947,hyper_SBC4,SBC11,SBC4,NC_012876.2,63484731,63485968
4,LOC110433273,-0.562994,0.562994,hyper_SBC23,SBC11,SBC23,NC_012872.2,17366045,17366207
5,LOC8074345,-0.554586,0.554586,hyper_SBC23,SBC11,SBC23,NC_012874.2,65779128,65779445
6,LOC8079764,-0.540058,0.540058,hyper_SBC23,SBC11,SBC23,NC_012870.2,22449132,22449709
7,LOC8064491,-0.533174,0.533174,hyper_SBC23,SBC11,SBC23,NC_012870.2,13890142,13890569
8,LOC8062471,-0.530691,0.530691,hyper_SBC23,SBC11,SBC23,NC_012870.2,13225228,13227500
9,LOC8077906,-0.512978,0.512978,hyper_SBC23,SBC11,SBC23,NC_012878.2,45590629,45590874


### Promoter hypomethylated in SBC11 across the dry-vs-juicy contrasts

In [17]:
# --- Refinement: hypomethylated in SBC11 CONSISTENTLY across dry-vs-juicy ------
# Strongest candidates are hypomethylated in SBC11 in EVERY SBC11-vs-juicy
# contrast where the gene's promoter appears -- not just one -- so the signal
# tracks the 3-vs-1 phenotype split rather than a single accession pair.
JUICY = ["SBC4", "SBC10", "SBC23"]

dry_vs_juicy = promoter_dmr[
    ((promoter_dmr["sample_a"] == SBC11) & promoter_dmr["sample_b"].isin(JUICY))
    | ((promoter_dmr["sample_b"] == SBC11) & promoter_dmr["sample_a"].isin(JUICY))
].copy()
dry_vs_juicy["hypo_in_dry"] = dry_vs_juicy["direction"] != f"hyper_{SBC11}"

consistency = (
    dry_vs_juicy.groupby("gene_label")
    .agg(n_contrasts=("direction", "size"),
         n_hypo_dry=("hypo_in_dry", "sum"),
         mean_diff=("diff.Methy", "mean"))
)
# keep genes hypomethylated in SBC11 in every dry-vs-juicy contrast they appear in
consistency["all_hypo_dry"] = consistency["n_hypo_dry"] == consistency["n_contrasts"]

phenotype_candidates = (
    consistency[consistency["all_hypo_dry"] & (consistency["n_contrasts"] >= 2)]
    .sort_values("mean_diff", key=lambda s: s.abs(), ascending=False)
    .reset_index()
)
print(f"{len(phenotype_candidates)} genes with a promoter hypomethylated in {SBC11} "
      f"across >=2 dry-vs-juicy contrasts")
phenotype_candidates.head(20)

1204 genes with a promoter hypomethylated in SBC11 across >=2 dry-vs-juicy contrasts


,gene_label,n_contrasts,n_hypo_dry,mean_diff,all_hypo_dry
0,LOC8080241,2,2,-0.442617,True
1,LOC8079764,2,2,-0.414391,True
2,LOC8062471,2,2,-0.371470,True
3,LOC8084237,2,2,-0.356687,True
4,LOC110431956,2,2,-0.330738,True
5,LOC110432782,2,2,-0.330057,True
6,LOC8068093,2,2,0.297456,True
7,LOC8072946,2,2,-0.283170,True
8,LOC8080209,2,2,-0.282329,True
9,LOC8084307,2,2,0.274740,True


### Positive-control check: is the D-gene's promoter hypomethylated in SBC11?

In [18]:
# --- Positive control: is the D-gene a SBC11-hypomethylated promoter candidate? --
DGENE_EGI = "8084661"   # D-gene; gene_label in DMR table is "LOC<EGI>"
DGENE = f"LOC{DGENE_EGI}"

in_candidates = DGENE in set(promoter_candidates["gene_label"])
print(f"{DGENE} hypomethylated-in-{SBC11} promoter candidate? {in_candidates}")

# show ALL of its promoter DMRs (any direction) for context
hit = promoter_dmr[promoter_dmr["gene_label"] == DGENE]
if hit.empty:
    print(f"{DGENE}: no promoter DMR in any contrast")
else:
    display(hit[["diff.Methy", "direction", "sample_a", "sample_b",
                 "chr", "start", "end"]])

LOC8084661 hypomethylated-in-SBC11 promoter candidate? False
LOC8084661: no promoter DMR in any contrast


## Important point
D-gene is not found to be mutated in the juicy variants nor DMR analysis, indicating that:
> Juiciness phenotype observed among the samples (SBC4, 10, 23) is not from D-gene mutation/LoF nor repressed expression by methylation, but rather from a set of genes regulated by the D-gene TF: dPCD-related genes set. <br>

Note: Additional step is needed to check whether D-gene mutation is observed in other variant groups.

## Switching to Co-expression file for finding dPCD-related genes

### dPCD related genes from literature review
| Generic Gene name | Arabidopsis gene ID, as found in the dPCD paper | Homolog in sorghum |
|-------------------|-------------------------------------------------|--------------------|
| PASPA3 - Aspartic protease 3 | At4g04460 | LOC8058198	LOC110430277	LOC110433911 |
| CEP1 - Cysteine endopeptidase | At5g50260 | LOC8058001 |
| MC9 - Metacaspase | At5g04200 | N/A |
| SCPL48 - Serine carboxypeptidase-like | At3g45010 | LOC8074232 |
| BFN1 - Bifunctional nuclease | At1g11190 | N/A |
| RNS3 - T2 Ribonuclease | At1g26820 | LOC8063515	LOC8063317	LOC8077825 | 

- Based on the Orvela-Carillo (2015) paper. Interestingly, the paper also mentioned that NAC family TFs are also included in this genes set. Utilizing this information, in the TF gene filtering, we will only take into account NAC family TF from the 2000 genes.
- Homolog(s) in sorghum is detected by ATTED-II built in orthologs tab

In [20]:
ROOT    = Path("/Users/daffa/workspace/infobio")
ZD      = ROOT / "Sbi-r.c1-0" / "Sbi-r.v25-12.G21627-S807.combat_pca.subagging.z.d"
TF_NAC  = ROOT / "thesis" / "resources" / "TFDB" / "Sbi_TF_list_NAC.txt"

SEN102  = "8058001"          # bait — the ONLY gene id hard-coded here
# Candidate universe restricted to the NAC family (Olvera-Carrillo 2015: NAC TFs
# are part of the dPCD gene set). D (8084661) / chr6 stay absent — keep it that way.

assert (ZD / SEN102).exists(), "SEN102 neighborhood file missing"
print("resource :", ZD.name)
print("bait     : SEN102 =", SEN102)

resource : Sbi-r.v25-12.G21627-S807.combat_pca.subagging.z.d
bait     : SEN102 = 8058001


### Checking seed-set purity: are these homologs in sorghum really co-expressed together?
Before aggregating co-expression across the 8 dPCD homologs, check they actually
co-express. We compute the **symmetric Mutual-Rank** between every seed pair
(`MR = sqrt(r_fwd · r_rev)`, lower = tighter, out of ~21626 genes) and then
**exhaustively enumerate every subset that contains the validated bait CEP1/SEN102**
(2⁷ = 128 subsets). Each subset is scored by its mean pairwise MR and compared to a
**size-matched random null** (random gene k-subsets), giving a z-score and empirical
p — so "which seeds belong together" is a number, not an eyeballed dendrogram cut.

In [21]:
import itertools

# --- dPCD seed genes (EGI). CEP1/SEN102 is the original validated bait. ----------
SEEDS = {
    "8058198":   "PASPA3-a", "110430277": "PASPA3-b", "110433911": "PASPA3-c",
    "8058001":   "CEP1/SEN102", "8074232": "SCPL48",
    "8063515":   "RNS3-a", "8063317": "RNS3-b", "8077825": "RNS3-c",
}
ANCHOR = "8058001"                      # CEP1/SEN102 — present in every subset tested
RNG    = np.random.default_rng(0)

def load_ranks(egi):
    """{partner_egi: 1-based rank within egi's genome-wide neighborhood}, N_total"""
    d = {}
    with open(ZD / egi) as fh:
        for i, line in enumerate(fh, 1):
            g, _ = line.split()
            d[g] = i
    return d, i

rank, Nrec = {}, {}
for g in SEEDS:
    rank[g], Nrec[g] = load_ranks(g)
ALL_GENES = list(rank[ANCHOR].keys()) + [ANCHOR]      # genome-wide universe

def smr(a, b, rk):
    ra = rk[a].get(b, Nrec.get(a, len(rk[a])) + 1)
    rb = rk[b].get(a, Nrec.get(b, len(rk[b])) + 1)
    return np.sqrt(ra * rb)                            # symmetric Mutual Rank

def coherence(members, rk):                            # mean pairwise MR (lower = tighter)
    return float(np.mean([smr(a, b, rk) for a, b in itertools.combinations(members, 2)]))

# --- size-matched random null: load a gene pool once, draw random k-subsets -------
POOL_SIZE, N_PERM = 300, 3000
pool  = list(RNG.choice(ALL_GENES, size=POOL_SIZE, replace=False))
prank = {}
for g in pool:
    prank[g], Nrec[g] = load_ranks(g)

null = {k: np.array([coherence(RNG.choice(pool, size=k, replace=False), prank)
                     for _ in range(N_PERM)]) for k in range(2, len(SEEDS) + 1)}
null_mean = {k: v.mean() for k, v in null.items()}
null_std  = {k: v.std()  for k, v in null.items()}

# --- exhaustive enumeration of every CEP1-containing subset (size >= 2) -----------
others = [g for g in SEEDS if g != ANCHOR]
rows = []
for r in range(1, len(others) + 1):
    for combo in itertools.combinations(others, r):
        members = (ANCHOR,) + combo
        k, obs  = len(members), coherence((ANCHOR,) + combo, rank)
        rows.append({
            "k": k,
            "members": ", ".join(SEEDS[m] for m in members),
            "mean_pairwise_MR": round(obs, 1),
            "z_vs_random": round((obs - null_mean[k]) / null_std[k], 2),   # neg = tighter
            "emp_p": round((np.sum(null[k] <= obs) + 1) / (N_PERM + 1), 4),
        })

seed_subsets = pd.DataFrame(rows).sort_values("z_vs_random").reset_index(drop=True)
print("Most coherent CEP1-containing seed subsets (lowest z = tightest vs size-matched random):\n")
seed_subsets.head(12)

Most coherent CEP1-containing seed subsets (lowest z = tightest vs size-matched random):



,k,members,mean_pairwise_MR,z_vs_random,emp_p
0,4,"CEP1/SEN102, PASPA3-a, RNS3-b, RNS3-c",2478.6,-3.27,0.0013
1,5,"CEP1/SEN102, PASPA3-a, RNS3-a, RNS3-b, RNS3-c",4826.3,-3.05,0.0033
2,4,"CEP1/SEN102, PASPA3-a, RNS3-a, RNS3-c",3343.8,-2.93,0.0037
3,3,"CEP1/SEN102, PASPA3-a, RNS3-c",853.4,-2.76,0.0010
4,3,"CEP1/SEN102, RNS3-b, RNS3-c",1899.0,-2.47,0.0050
5,5,"CEP1/SEN102, PASPA3-a, PASPA3-c, RNS3-b, RNS3-c",6234.5,-2.33,0.0160
6,5,"CEP1/SEN102, PASPA3-b, PASPA3-c, SCPL48, RNS3-b",6503.8,-2.19,0.0210
7,6,"CEP1/SEN102, PASPA3-a, PASPA3-c, RNS3-a, RNS3-...",7081.2,-2.18,0.0250
8,4,"CEP1/SEN102, RNS3-a, RNS3-b, RNS3-c",5340.4,-2.14,0.0237
9,3,"CEP1/SEN102, RNS3-a, RNS3-c",3488.0,-2.03,0.0267


In [22]:
# Where the eyeballed "Module A" {CEP1, RNS3-c, PASPA3-a} lands, and the global optimum.
core = {"CEP1/SEN102", "RNS3-c", "PASPA3-a"}
is_core = seed_subsets["members"].apply(lambda s: set(s.split(", ")) == core)
print("Eyeballed 'Module A' {CEP1, RNS3-c, PASPA3-a}:")
print(seed_subsets[is_core].to_string(index=False))
print(f"\nGlobal best subset by size-normalised z: "
      f"{seed_subsets.iloc[0]['members']}  "
      f"(z={seed_subsets.iloc[0]['z_vs_random']}, p={seed_subsets.iloc[0]['emp_p']})")

# Which seeds never help the CEP1 module? Mean z of subsets containing each seed.
contrib = {SEEDS[g]: seed_subsets[seed_subsets["members"].str.contains(SEEDS[g], regex=False)]["z_vs_random"].mean()
           for g in others}
print("\nMean z of all CEP1-subsets containing each candidate seed (more negative = pulls the module tighter):")
for name, mz in sorted(contrib.items(), key=lambda kv: kv[1]):
    print(f"  {name:10s} {mz:6.2f}")

Eyeballed 'Module A' {CEP1, RNS3-c, PASPA3-a}:
 k                       members  mean_pairwise_MR  z_vs_random  emp_p
 3 CEP1/SEN102, PASPA3-a, RNS3-c             853.4        -2.76  0.001

Global best subset by size-normalised z: CEP1/SEN102, PASPA3-a, RNS3-b, RNS3-c  (z=-3.27, p=0.0013)

Mean z of all CEP1-subsets containing each candidate seed (more negative = pulls the module tighter):
  RNS3-b      -1.41
  RNS3-c      -1.33
  PASPA3-c    -1.01
  RNS3-a      -1.01
  PASPA3-a    -0.99
  PASPA3-b    -0.79
  SCPL48      -0.73


The dPCD genes that are confirmed to be tightly co-expressed are:
{CEP1/SEN102: "8058001", RNS3-c: "8077825", PASPA3-a: "8058198"}

## Work-around: finding these dPCD gene module in the variant & DMR genes lists

In [26]:
# The co-expression-validated dPCD core (from the seed-set hygiene step above)
MODULE = {
    "LOC8058001": "CEP1/SEN102",
    "LOC8077825": "RNS3-c",
    "LOC8058198": "PASPA3-a",
}

# 1) Variant evidence -- rank of the module under each gene-merging method
print("=== Variant gene-score rank of the dPCD module (SBC4_SBC10_SBC23 group) ===")
for name, df in merged_dfs.items():
    print(f"\n[{name}]")
    display(where_genes(df, MODULE))

# 2) DMR evidence -- is any module gene a SBC11-hypomethylated promoter candidate?
print("\n=== DMR promoter evidence (SBC11-hypomethylated candidates) ===")
dmr_hit = promoter_candidates[promoter_candidates["gene_label"].isin(MODULE)].copy()
if dmr_hit.empty:
    print("no module gene among SBC11-hypomethylated promoter candidates")
else:
    dmr_hit.insert(0, "gene", dmr_hit["gene_label"].map(MODULE))
    display(dmr_hit)

=== Variant gene-score rank of the dPCD module (SBC4_SBC10_SBC23 group) ===

[gene_max]


,gene,found,score,percentile,rank
ann_gene_id,,,,,
LOC8058001,CEP1/SEN102,True,0.375,91.553766,91.0
LOC8077825,RNS3-c,False,NaN,NaN,NaN
LOC8058198,PASPA3-a,False,NaN,NaN,NaN



[gene_sum]


,gene,found,score,percentile,rank
ann_gene_id,,,,,
LOC8058001,CEP1/SEN102,True,5.125,91.000232,485.0
LOC8077825,RNS3-c,False,NaN,NaN,NaN
LOC8058198,PASPA3-a,False,NaN,NaN,NaN



[gene_max_norm]


,gene,found,score,percentile,rank
ann_gene_id,,,,,
LOC8058001,CEP1/SEN102,True,0.226723,90.70827,617.0
LOC8077825,RNS3-c,False,NaN,NaN,NaN
LOC8058198,PASPA3-a,False,NaN,NaN,NaN



[gene_sum_norm]


,gene,found,score,percentile,rank
ann_gene_id,,,,,
LOC8058001,CEP1/SEN102,True,3.098549,94.408394,405.0
LOC8077825,RNS3-c,False,NaN,NaN,NaN
LOC8058198,PASPA3-a,False,NaN,NaN,NaN



=== DMR promoter evidence (SBC11-hypomethylated candidates) ===
no module gene among SBC11-hypomethylated promoter candidates


# Conclusion
- Based on the genomics data (variant analysis), D-gene, the transcription factor that causes the death of pith parenchyma cells hence the causative for dry stem phenotype, is not found to be mutated/having LoF in the juicy stem samples' genome
- Based on the epigenomics data (DMR analysis), D-gene was also not found to be repressed in juicy stem samples' methylome
- It is inferred that the juicy phenotype observed is caused by mutation in the dPCD genes, that are genes upregulated by the D-gene transcription factor
- We obtained a list of dPCD genes in model species from literature review, finding the sorghum homologs, and improving its 'purity' by referring to the gene co-expression data in sorghum
- Three out of eight dPCD genes are tightly co-expressed: LOC8058001, LOC8077825, LOC8058198
- Going back to the sample data, out of three dPCD genes, only LOC8058001 is found to contain variant. 
- Meanwhile, Epigenomics (DMR analysis) data doesn't have information on dPCD genes

In [32]:
df_scored_gene_max[df_scored_gene_max["ann_gene_id"] == "LOC8058001"]

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score,percentile,rank
980,LOC8058001,LOC8058001,protein_coding,NC_012876.2,0.375,91.553766,91


In [ ]:
# full annotation on the variant site
df_scored[(df_scored["ann_gene_id"] == "LOC8058001") & (df_scored["ann_impact"] != "MODIFIER")]

,chrom,pos,ref,alt,qual,filter,gt,ann_allele,ann_effect,ann_impact,...,sift_variant_type,sift_aa_change,sift_aa_pos,sift_score,sift_median,sift_num_seqs,sift_allele_type,sift_prediction,sift_score_c,score
116460,NC_012876.2,60705469,A,G,49.25,NaN,1/1,G,synonymous_variant,LOW,...,SYNONYMOUS,S/S,272.0,0.69,3.27,397.0,novel,TOLERATED,0.31,0.375
